# 📚 Assignment 1 — SocratesAI: 소크라테스식 1:1 학습 코치

---

## Step 1: 에이전트 설계

### 이름
**SocratesAI** — 소크라테스식 1:1 학습 코치

### 목적
학습자가 원하는 개념을 입력하면, 현재 이해도를 먼저 진단하고, 그 수준에 맞는 맞춤형 설명을 제공한 뒤, 퀴즈로 이해를 검증하고 피드백을 반복하여 개념을 완전히 이해할 때까지 도와주는 개인 맞춤형 AI 학습 코치.

### 핵심 기능 (3가지 이상)

| # | 기능 | 설명 |
|---|------|------|
| 1 | **이해도 자동 진단** | 학습자의 배경 지식을 분석해 beginner / intermediate / advanced 레벨을 자동 결정 |
| 2 | **레벨 맞춤형 설명** | 진단된 레벨에 맞게 비유·예시·깊이를 달리하여 개념 설명 생성 |
| 3 | **이해도 퀴즈** | 설명 내용 기반으로 단답형/서술형 퀴즈 자동 출제 |
| 4 | **채점 & 피드백 루프** | 답변을 채점하고, 70점 미만이면 재설명 (최대 3회 반복) |

### 그래프 구조 다이어그램

```
                    START
                      │
                      ▼
              ┌───────────────┐
              │    assess     │  ← 배경지식 분석 → 레벨 결정
              └───────┬───────┘
                      │
          ┌───────────▼───────────┐
          │       explain         │  ← 레벨 맞춤 개념 설명
          └───────────┬───────────┘
                      │          ▲
                      ▼          │ retry (score < 70, attempt < 3)
              ┌───────────────┐  │
              │     quiz      │  │
              └───────┬───────┘  │
                      │          │
              ┌───────▼───────┐  │
              │   evaluate    │──┘
              └───────┬───────┘
                      │
          ┌───────────┴───────────┐
          │                       │
    score >= 70              attempt >= 3
    또는 통과                또는 탈출 조건
          │                       │
          └───────────┬───────────┘
                      │
                     END
```

**조건부 엣지 (Conditional Edge)**:
- `score >= 70` → **END** ✅ (학습 완료)
- `score < 70` AND `attempt_count < 3` → **explain** (재설명 루프)
- `attempt_count >= 3` → **END** ⏱️ (최대 시도 초과)


---

## Step 2: 기초 구축

### 2-1. 패키지 설치


In [ ]:
%pip install langgraph langchain-openai python-dotenv -q

### 2-2. 환경 설정 및 임포트


In [ ]:
import os
import json
import re
from typing import TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()

# API 키 확인
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env에 설정되어 있어야 합니다."
print("✅ 환경 설정 완료")

---

### 2-3. State 정의

모든 에이전트 노드가 공유하는 상태(State)를 정의합니다.  
각 노드는 자신이 담당하는 필드만 업데이트합니다.


In [ ]:
class LearningState(TypedDict, total=False):
    # ── 입력 ──────────────────────────────────────
    topic: str          # 학습할 주제 (예: "LangGraph 조건부 엣지")
    background: str     # 학습자의 배경 지식 (선택 입력)

    # ── 진단 (assess 노드) ─────────────────────────
    level: str          # "beginner" | "intermediate" | "advanced"

    # ── 학습 콘텐츠 (explain 노드) ─────────────────
    explanation: str    # 생성된 레벨 맞춤 설명

    # ── 퀴즈 (quiz 노드) ───────────────────────────
    quiz_question: str  # 출제된 문제 (학습자에게 공개)
    quiz_hint: str      # 채점용 핵심 키워드 (학습자에게 비공개)
    quiz_answer: str    # 학습자의 답변

    # ── 평가 (evaluate 노드) ───────────────────────
    score: int          # 점수 (0~100)
    feedback: str       # 피드백 메시지

    # ── 메타 ──────────────────────────────────────
    attempt_count: int  # 현재까지 시도한 횟수

print("✅ State 정의 완료")
print("필드:", list(LearningState.__annotations__.keys()))

---

### 2-4. 노드 구현

#### 노드 1: `assess` — 이해도 진단

학습자의 배경 지식을 파악하고 적절한 학습 레벨을 결정합니다.


In [ ]:
def assess_node(state: LearningState) -> dict:
    """노드 1 — 학습자 수준을 진단하고 레벨을 결정한다."""
    topic = state["topic"]
    background = state.get("background", "")

    print(f"\n🔍 [Assess] 주제: '{topic}'")
    print(f"   배경 지식: {background or '(없음)'}")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

    prompt = f"""학습 주제: {topic}
학습자 배경: {background if background else '없음 (초보자로 가정)'}

위 정보를 바탕으로 학습자에게 가장 적합한 레벨을 결정하세요.
- beginner: 기초 개념 위주, 비유 많이 사용, 전문 용어 최소화
- intermediate: 개념 + 작동 원리 설명, 적당한 기술 용어 사용
- advanced: 깊은 원리, 엣지 케이스, 실무 적용까지

반드시 아래 JSON만 응답하세요 (다른 텍스트 없이):
{{"level": "beginner또는intermediate또는advanced", "reason": "한 줄 이유"}}"""

    response = llm.invoke([
        SystemMessage(content="당신은 교육 전문가입니다. 반드시 JSON만 응답하세요."),
        HumanMessage(content=prompt),
    ])

    try:
        text = response.content.strip()
        # 마크다운 코드블록 제거
        if text.startswith("```"):
            text = re.sub(r"```[\w]*\n?", "", text).strip()
        result = json.loads(text)
        level = result.get("level", "beginner")
        reason = result.get("reason", "")
    except (json.JSONDecodeError, Exception):
        level = "beginner"
        reason = "파싱 실패 — 기본값 적용"

    level_emoji = {"beginner": "🌱", "intermediate": "🌿", "advanced": "🌳"}
    print(f"   → 레벨 결정: {level_emoji.get(level, '')} {level} ({reason})")

    return {"level": level}

print("✅ assess_node 정의 완료")

#### 노드 2: `explain` — 맞춤형 개념 설명

결정된 레벨에 맞게 개념을 설명합니다. 재시도 시에는 이전 피드백을 반영합니다.


In [ ]:
def explain_node(state: LearningState) -> dict:
    """노드 2 — 레벨에 맞춘 개념 설명을 생성한다. 재시도 시 피드백 반영."""
    topic = state["topic"]
    level = state.get("level", "beginner")
    attempt = state.get("attempt_count", 0)
    prev_feedback = state.get("feedback", "")

    print(f"\n📖 [Explain] 설명 생성 중... (시도 #{attempt + 1}, 레벨: {level})")

    level_guide = {
        "beginner": "초보자를 위해 일상적인 비유를 적극 활용하고, 전문 용어는 쉬운 말로 바꿔 설명하세요.",
        "intermediate": "기본 개념은 알고 있다고 가정하고, 작동 원리와 내부 메커니즘을 설명하세요.",
        "advanced": "깊은 원리, 엣지 케이스, 실무에서의 주의사항까지 포함하여 심화 설명하세요.",
    }

    retry_context = ""
    if attempt > 0 and prev_feedback:
        retry_context = f"\n\n[재설명 요청] 이전 설명에서 학습자가 부족했던 부분:\n{prev_feedback}\n이 부분을 특히 더 명확하게 설명해주세요."

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

    response = llm.invoke([
        SystemMessage(content=f"당신은 친절하고 명확한 교육 전문가입니다. {level_guide[level]}"),
        HumanMessage(content=f"""다음 개념을 설명해주세요: {topic}{retry_context}

형식:
1. 핵심 개념 설명 (3~4 문단)
2. 실생활 비유 또는 예시 1개
3. 핵심 포인트 3가지 (bullet point)"""),
    ])

    print("   → 설명 생성 완료 ✅")
    return {"explanation": response.content}

print("✅ explain_node 정의 완료")

#### 노드 3: `quiz` — 이해도 퀴즈 출제

방금 설명한 내용을 바탕으로 단답형/서술형 문제를 출제합니다.


In [ ]:
def quiz_node(state: LearningState) -> dict:
    """노드 3 — 이해도 확인 퀴즈를 생성하고 학습자에게 출제한다."""
    topic = state["topic"]
    explanation = state.get("explanation", "")
    level = state.get("level", "beginner")

    print(f"\n❓ [Quiz] 문제 생성 중... (레벨: {level})")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)

    response = llm.invoke([
        SystemMessage(content="당신은 교육 평가 전문가입니다. 이해도를 측정하는 좋은 문제를 만드세요."),
        HumanMessage(content=f"""주제: {topic} (레벨: {level})
방금 한 설명 (앞 800자):
{explanation[:800]}

위 설명을 이해했는지 확인하는 서술형 문제 1개를 만드세요.

반드시 아래 형식으로만 응답하세요:
[문제] 문제 내용을 여기에 작성
[힌트키워드] 채점 기준이 될 핵심 단어 3개 (쉼표 구분)"""),
    ])

    raw = response.content
    question = raw
    hint = ""

    if "[힌트키워드]" in raw:
        parts = raw.split("[힌트키워드]")
        question = parts[0].replace("[문제]", "").strip()
        hint = parts[1].strip()
    elif "[문제]" in raw:
        question = raw.replace("[문제]", "").strip()

    print(f"\n{'='*60}")
    print(f"📝 퀴즈 문제")
    print(f"{'='*60}")
    print(question)
    print(f"{'='*60}")

    # 실제 사용 시: quiz_answer = input("\n답변을 입력하세요: ")
    # 데모에서는 state에 미리 설정된 quiz_answer를 사용

    return {"quiz_question": question, "quiz_hint": hint}

print("✅ quiz_node 정의 완료")

#### 노드 4: `evaluate` — 채점 & 피드백 생성

학습자의 답변을 채점하고, 조건부 엣지의 분기 기준이 될 점수와 피드백을 생성합니다.


In [ ]:
def evaluate_node(state: LearningState) -> dict:
    """노드 4 — 답변을 채점하고 피드백을 생성한다."""
    quiz = state.get("quiz_question", "")
    answer = state.get("quiz_answer", "")
    hint = state.get("quiz_hint", "")
    attempt = state.get("attempt_count", 0)

    print(f"\n📊 [Evaluate] 채점 중... (시도 #{attempt + 1})")
    print(f"   학습자 답변: {answer[:100]}{'...' if len(answer) > 100 else ''}")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    response = llm.invoke([
        SystemMessage(content="당신은 공정하고 격려적인 교육 평가자입니다."),
        HumanMessage(content=f"""문제: {quiz}
채점 기준 키워드: {hint}
학습자 답변: {answer}

아래 형식으로 채점하세요:
[점수] 0~100 사이 숫자만
[피드백] 잘한 점 + 보완할 점 (2~3문장, 친절하게)
[부족한개념] 재설명 시 강조할 핵심 개념 (없으면 '없음')"""),
    ])

    content = response.content
    score = 50  # 파싱 실패 시 기본값
    feedback = content
    weak_concept = ""

    score_match = re.search(r"\[점수\]\s*(\d+)", content)
    if score_match:
        score = int(score_match.group(1))

    feedback_match = re.search(r"\[피드백\](.*?)(?=\[|$)", content, re.DOTALL)
    if feedback_match:
        feedback = feedback_match.group(1).strip()

    weak_match = re.search(r"\[부족한개념\](.*?)(?=\[|$)", content, re.DOTALL)
    if weak_match:
        weak_concept = weak_match.group(1).strip()

    if weak_concept and weak_concept != "없음" and feedback:
        feedback = feedback + f"\n\n[재설명 시 집중할 부분]: {weak_concept}"

    result_emoji = "✅" if score >= 70 else "🔄"
    print(f"   → 점수: {score}점 {result_emoji}")
    print(f"   → 피드백: {feedback[:120]}...")

    return {
        "score": score,
        "feedback": feedback,
        "attempt_count": attempt + 1,
    }


def should_retry(state: LearningState) -> str:
    """조건부 엣지 함수 — 채점 결과에 따라 다음 노드를 결정한다."""
    score = state.get("score", 0)
    attempt = state.get("attempt_count", 0)

    if score >= 70:
        print(f"\n🎉 통과! (점수: {score}) → 학습 완료")
        return "pass"
    elif attempt >= 3:
        print(f"\n⏱️ 최대 시도 횟수 초과 (3회) → 종료")
        return "max_attempts"
    else:
        print(f"\n🔄 재도전! (점수: {score}, 시도: {attempt}/3) → 재설명")
        return "retry"


print("✅ evaluate_node + should_retry 정의 완료")

---

### 2-5. 그래프 구성 (LangGraph)

4개의 노드와 조건부 엣지를 연결하여 학습 루프 그래프를 완성합니다.


In [ ]:
def build_graph() -> StateGraph:
    """SocratesAI LangGraph 워크플로우를 구성한다."""
    workflow = StateGraph(LearningState)

    # ── 노드 등록 ──────────────────────────────────
    workflow.add_node("assess",   assess_node)
    workflow.add_node("explain",  explain_node)
    workflow.add_node("quiz",     quiz_node)
    workflow.add_node("evaluate", evaluate_node)

    # ── 일반 엣지 (순서대로 연결) ──────────────────
    workflow.add_edge(START,     "assess")
    workflow.add_edge("assess",  "explain")
    workflow.add_edge("explain", "quiz")
    workflow.add_edge("quiz",    "evaluate")

    # ── 조건부 엣지: evaluate → 다음 노드 결정 ─────
    workflow.add_conditional_edges(
        "evaluate",
        should_retry,
        {
            "pass":          END,       # 70점 이상 → 학습 완료
            "retry":         "explain", # 70점 미만 + 시도 여유 → 재설명
            "max_attempts":  END,       # 3회 초과 → 강제 종료
        },
    )

    return workflow


# 컴파일
app = build_graph().compile()
print("✅ 그래프 컴파일 완료")
print("\n노드 목록:", list(app.nodes.keys()))

In [ ]:
# 그래프 시각화 (Mermaid 다이어그램)
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("시각화 불가 (선택사항):", e)
    print("\n그래프 구조 (Mermaid):")
    print(app.get_graph().draw_mermaid())

---

## 데모 실행

### 시나리오 A — 초보자가 'LangGraph 조건부 엣지'를 처음 배우는 경우

> 📌 `quiz_answer`에 사전 답변을 설정하여 전체 파이프라인을 end-to-end로 테스트합니다.  
> 실제 서비스에서는 `quiz_answer`를 UI 입력 또는 `input()`으로 받습니다.


In [ ]:
# 시나리오 A: 초보자 — 정답을 잘 맞추는 경우 (통과 시나리오)
initial_state_a = {
    "topic": "LangGraph의 조건부 엣지(conditional edge)란 무엇인가요?",
    "background": "Python은 알지만 LangGraph는 처음 배웁니다.",
    "quiz_answer": (
        "조건부 엣지는 그래프에서 특정 노드가 실행된 후 "
        "조건 함수의 반환값에 따라 다음에 실행할 노드를 동적으로 결정하는 연결 방식입니다. "
        "예를 들어 에디터 노드가 품질을 평가한 뒤 '통과'면 출판 노드로, "
        "'수정 필요'면 작성 노드로 흐름을 분기할 수 있습니다."
    ),
    "attempt_count": 0,
}

print("=" * 60)
print("🚀 SocratesAI 데모 시작 — 시나리오 A (통과)")
print("=" * 60)

result_a = app.invoke(initial_state_a)

print("\n" + "=" * 60)
print("📊 최종 결과 요약")
print("=" * 60)
print(f"레벨        : {result_a.get('level')}")
print(f"최종 점수   : {result_a.get('score')}점")
print(f"총 시도 횟수: {result_a.get('attempt_count')}회")
print(f"\n[피드백]")
print(result_a.get('feedback', ''))

In [ ]:
# 시나리오 B: 중급자 — 처음엔 부족하게 답하다가 재설명 후 통과하는 경우
# quiz_answer가 짧아서 첫 번째 시도에서 낮은 점수를 받고 재설명을 받는 시나리오
initial_state_b = {
    "topic": "Python 제너레이터(Generator)와 일반 함수의 차이",
    "background": "Python을 6개월 정도 사용했습니다. 함수와 리스트는 잘 압니다.",
    "quiz_answer": "yield를 쓰는 함수입니다.",  # 짧고 불완전한 답 → 재설명 유도
    "attempt_count": 0,
}

print("=" * 60)
print("🚀 SocratesAI 데모 시작 — 시나리오 B (재설명 루프)")
print("=" * 60)
print("📌 NOTE: quiz_answer가 짧아 재설명 루프가 작동하는 시나리오입니다.")
print("         (attempt_count >= 3 도달 시 자동 종료됩니다)\n")

result_b = app.invoke(initial_state_b)

print("\n" + "=" * 60)
print("📊 최종 결과 요약")
print("=" * 60)
print(f"레벨        : {result_b.get('level')}")
print(f"최종 점수   : {result_b.get('score')}점")
print(f"총 시도 횟수: {result_b.get('attempt_count')}회")
print(f"\n[마지막 피드백]")
print(result_b.get('feedback', ''))

---

## 구현 요약

| 요구사항 | 구현 여부 |
|----------|-----------|
| LangGraph 사용 | ✅ `StateGraph` + 조건부 엣지 |
| 최소 2개 작동하는 노드 | ✅ 4개 노드 구현 (assess, explain, quiz, evaluate) |
| State 정의 | ✅ `LearningState(TypedDict)` — 11개 필드 |
| 설계 문서 | ✅ 이름, 목적, 핵심 기능, 그래프 다이어그램 |
| 조건부 엣지 (사이클) | ✅ evaluate → explain 재설명 루프 (최대 3회) |

### 핵심 LangGraph 개념 적용

- **TypedDict State**: 모든 노드가 공유하는 상태 스키마로 각 노드가 자기 담당 필드만 업데이트
- **Conditional Edge**: `should_retry` 함수가 점수 기준으로 3가지 분기 결정 (`pass` / `retry` / `max_attempts`)
- **Cycle (루프)**: `evaluate → explain → quiz → evaluate` 반복 구조로 진짜 학습 파이프라인 구현
- **탈출 조건**: `attempt_count >= 3` 강제 종료로 무한 루프 방지
